In [3]:
!rclone copy :s3:ndha-public-data-ap-southeast-2/iPRES-2025/sample-data/covid19.govt.nz/2023-12-14_IE89493927/FL89493929_NLNZ-20231212233435565-00000-72544~wlgprdwctweb01.natlib.govt.nz~8443.warc.gz .

2025/10/24 13:44:21 NOTICE: s3: s3 provider "" not known - please set correctly


2025/10/24 13:44:21 NOTICE: s3: s3 provider "" not known - please set correctly
2025/10/24 13:44:21 NOTICE: S3 bucket ndha-public-data-ap-southeast-2 path iPRES-2025/sample-data/covid19.govt.nz/2023-12-14_IE89493927: Switched region to "ap-southeast-2" from "us-east-1"


In [10]:
!rclone copy :s3:aws-publicdatasets/common-crawl/crawl-data/CC-MAIN-2013-48/segments/1386163035819/warc/CC-MAIN-20131204131715-00000-ip-10-33-133-15.ec2.internal.warc.gz .

2025/10/24 15:20:37 NOTICE: s3: s3 provider "" not known - please set correctly


In [1]:
!curl -O https://sembiance.com/fileFormatSamples/archive/warc/radius.vintagebox.de.warc

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 93.0M  100 93.0M    0     0  10.0M      0  0:00:09  0:00:09 --:--:-- 13.1M


In [27]:
import json
import pandas as pd
from subprocess import Popen, PIPE, STDOUT
from warcio.archiveiterator import ArchiveIterator

input_file = 'radius.vintagebox.de.warc'
#input_file = 'FL89493929_NLNZ-20231212233435565-00000-72544~wlgprdwctweb01.natlib.govt.nz~8443.warc.gz'

# sf -json -
def sf(payload):
    p = Popen(["sf", "-json", "-"], stdout=PIPE, stdin=PIPE, stderr=PIPE)
    stdout_data = p.communicate(input=payload)[0]
    result = json.loads(stdout_data)
    matches = result['files'][0]['matches']
    # Return an empty result for no matches
    if len(matches) == 0:
        matches.append({})
    return matches

results = []
with open(input_file, 'rb') as stream:
    for record in ArchiveIterator(stream):
        if record.rec_type == 'response':
            payload = record.content_stream().read()
            matches = sf(payload)
            for match in matches:
                match['uri'] = record.rec_headers.get_header('WARC-Target-URI')
                match['date'] = record.rec_headers.get_header('WARC-Date')
                match['warc_content_type'] = record.content_type
                match['warc_length'] = record.length
                match['http_content_type'] = record.http_headers.get_header('Content-Type', None)
                match['http_payload_length'] = len(payload)
                match['http_status_code'] = record.http_headers.get_statuscode()
                results.append(match)

df = pd.DataFrame(results)
df

,ns,id,format,version,mime,class,basis,warning,uri,date,warc_content_type,warc_length,http_content_type,http_payload_length,http_status_code
0,pronom,fmt/100,Hypertext Markup Language,4.01,text/html,Text (Mark-up),"byte match at 0, 44",,http://radius.vintagebox.de/,2018-11-20T11:17:01Z,application/http;msgtype=response,1038,text/html; charset=ISO-8859-1,743,200
1,pronom,x-fmt/111,Plain Text File,,text/plain,,text match ASCII,match on text only,http://radius.vintagebox.de/scripts/all.css,2018-11-20T11:17:01Z,application/http;msgtype=response,1664,text/css,1390,200
2,pronom,fmt/98,Hypertext Markup Language,3.2,text/html,Text (Mark-up),"byte match at 0, 43",,http://radius.vintagebox.de/navigation.html,2018-11-20T11:17:02Z,application/http;msgtype=response,2379,text/html; charset=ISO-8859-1,2084,200
3,pronom,fmt/100,Hypertext Markup Language,4.01,text/html,Text (Mark-up),"byte match at 0, 44",,http://radius.vintagebox.de/overview.html,2018-11-20T11:17:02Z,application/http;msgtype=response,3045,text/html; charset=ISO-8859-1,2750,200
4,pronom,fmt/4,Graphics Interchange Format,89a,image/gif,Image (Raster),byte match at [[0 6] [46 1]],,http://radius.vintagebox.de/img/nav/bg.gif,2018-11-20T11:17:02Z,application/http;msgtype=response,319,image/gif,47,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617,pronom,x-fmt/111,Plain Text File,,text/plain,,text match ASCII,match on text only,http://radius.vintagebox.de/download/radius/so...,2018-11-20T11:22:26Z,application/http;msgtype=response,91429,application/mac-binhex,91138,200
618,pronom,x-fmt/111,Plain Text File,,text/plain,,text match ASCII,match on text only,http://radius.vintagebox.de/download/radius/so...,2018-11-20T11:22:26Z,application/http;msgtype=response,701211,application/mac-binhex,700919,200
619,pronom,x-fmt/111,Plain Text File,,text/plain,,text match ASCII,match on text only,http://radius.vintagebox.de/download/radius/so...,2018-11-20T11:22:27Z,application/http;msgtype=response,46289,application/mac-binhex,45999,200
620,pronom,x-fmt/111,Plain Text File,,text/plain,,text match ASCII,match on text only,http://radius.vintagebox.de/download/radius/so...,2018-11-20T11:22:28Z,application/http;msgtype=response,317508,application/mac-binhex,317216,200


In [29]:
df.groupby(['format', 'version', 'id', 'http_content_type'])['uri'].count().reset_index()

,format,version,id,http_content_type,uri
0,Graphics Interchange Format,89a,fmt/4,image/gif,39
1,Hypertext Markup Language,2.0,fmt/97,text/html; charset=iso-8859-1,209
2,Hypertext Markup Language,3.2,fmt/98,text/html; charset=ISO-8859-1,194
3,Hypertext Markup Language,4.01,fmt/100,text/html; charset=ISO-8859-1,2
4,JPEG File Interchange Format,1.01,fmt/43,image/jpeg,14
5,JPEG File Interchange Format,1.02,fmt/44,image/jpeg,7
6,Plain Text File,,x-fmt/111,application/mac-binhex,104
7,Plain Text File,,x-fmt/111,text/css,1
8,Plain Text File,,x-fmt/111,text/html; charset=ISO-8859-1,1
9,Plain Text File,,x-fmt/111,text/plain; charset=ISO-8859-1,8
